In [1]:
import os
import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

from peft import PeftModel


# =============================================================================
# V5D MERGE CONFIGURATION
# =============================================================================

# Previous merged model becomes the base
BASE_MODEL = "./V5C_Final_Merged_Model"

# V5D trained LoRA adapter
V5D_LORA_PATH = "./V5D_Final/final_model/"

# New merged output
OUTPUT_PATH = "./V5D_Final_Merged_Model"


# =============================================================================
# CHECK PATHS
# =============================================================================

print("=" * 80)
print("V5D → FINAL MERGE")
print("=" * 80)

if not os.path.exists(BASE_MODEL):
    raise FileNotFoundError(
        f"V5C base model not found: {BASE_MODEL}"
    )

if not os.path.exists(V5D_LORA_PATH):
    raise FileNotFoundError(
        f"V5D LoRA adapter not found: {V5D_LORA_PATH}"
    )

os.makedirs(OUTPUT_PATH, exist_ok=True)


# =============================================================================
# LOAD TOKENIZER
# =============================================================================

print("\n" + "=" * 80)
print("1. LOADING TOKENIZER")
print("=" * 80)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
)

print("Tokenizer loaded.")


# =============================================================================
# LOAD V5C BASE MODEL
# =============================================================================

print("\n" + "=" * 80)
print("2. LOADING V5C BASE MODEL")
print("=" * 80)

print("Loading on CPU...")
print("This is intentional.")
print("RTX GPU is NOT used for the merge.")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,

    # Load in FP16 for LoRA merging
    torch_dtype=torch.float16,

    # CPU merge
    device_map="cpu",

    low_cpu_mem_usage=True,

    trust_remote_code=True,
)

print("V5C base model loaded successfully.")


# =============================================================================
# LOAD V5D LORA
# =============================================================================

print("\n" + "=" * 80)
print("3. LOADING V5D LORA ADAPTER")
print("=" * 80)

model = PeftModel.from_pretrained(
    base_model,
    V5D_LORA_PATH,
)

print("V5D LoRA loaded successfully.")


# =============================================================================
# MERGE
# =============================================================================

print("\n" + "=" * 80)
print("4. MERGING V5D LORA INTO V5C")
print("=" * 80)

print("Merging...")
print("Please wait.")

merged_model = model.merge_and_unload()

print("V5D LoRA merged successfully.")


# =============================================================================
# SAVE MERGED MODEL
# =============================================================================

print("\n" + "=" * 80)
print("5. SAVING V5D FINAL MERGED MODEL")
print("=" * 80)

print("Output:")
print(OUTPUT_PATH)

merged_model.save_pretrained(
    OUTPUT_PATH,
    safe_serialization=True,
    max_shard_size="4GB",
)

tokenizer.save_pretrained(
    OUTPUT_PATH
)

print("Model saved successfully.")


# =============================================================================
# CLEAN MEMORY
# =============================================================================

del model
del base_model
del merged_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# =============================================================================
# COMPLETE
# =============================================================================

print("\n" + "=" * 80)
print("✅ V5D MERGE COMPLETE")
print("=" * 80)

print()
print("Final model:")
print(OUTPUT_PATH)

print()
print("Pipeline:")
print("V5A → V5A Merged")
print("V5B → V5B Merged")
print("V5C → V5C Merged")
print("V5D → V5D Final Merged")

print("=" * 80)

g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


V5D → FINAL MERGE

1. LOADING TOKENIZER
Tokenizer loaded.

2. LOADING V5C BASE MODEL
Loading on CPU...
This is intentional.
RTX GPU is NOT used for the merge.


Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 11.87it/s]


V5C base model loaded successfully.

3. LOADING V5D LORA ADAPTER
V5D LoRA loaded successfully.

4. MERGING V5D LORA INTO V5C
Merging...
Please wait.
V5D LoRA merged successfully.

5. SAVING V5D FINAL MERGED MODEL
Output:
./V5D_Final_Merged_Model
Model saved successfully.

✅ V5D MERGE COMPLETE

Final model:
./V5D_Final_Merged_Model

Pipeline:
V5A → V5A Merged
V5B → V5B Merged
V5C → V5C Merged
V5D → V5D Final Merged
